#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("speech.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs=text_splitter.split_documents(documents)


c:\Users\VikashMahato\Udemy Learning\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embeddings  = OllamaEmbeddings(model="gemma:2b")
db = FAISS.from_documents(docs, embeddings)

In [4]:
db

In [6]:
### querying 
query="How does the speaker describe the desired outcome of the war?"
docs=db.similarity_search(query)
docs[0].page_content


'â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [8]:
retriever = db.as_retriever()
docs=  retriever.invoke(query)

In [9]:
docs[0].page_content

'â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [10]:

docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='e86e26d8-b8c2-4aa2-a0a8-9119745c9cac', metadata={'source': 'speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(2949.2173)),
 (Document(id='4ff56fbc-d94d-4009-ab77-667c7eb0a5c3', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of 

In [11]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[0.3096381425857544,
 -2.11273193359375,
 0.2457764893770218,
 1.0192961692810059,
 0.6540417671203613,
 0.5667218565940857,
 -0.901712954044342,
 0.09848184138536453,
 -0.045372117310762405,
 -0.8067281246185303,
 1.099103331565857,
 -0.03905283659696579,
 -1.1018433570861816,
 1.1030466556549072,
 0.060045573860406876,
 -1.0219695568084717,
 3.0162577629089355,
 1.7323274612426758,
 1.128972053527832,
 0.777755081653595,
 0.32309919595718384,
 -0.2713453471660614,
 0.2928799092769623,
 1.352156400680542,
 0.20632067322731018,
 -0.40175944566726685,
 -1.3564740419387817,
 -1.2929679155349731,
 -0.5493724942207336,
 -2.0747666358947754,
 -0.25916871428489685,
 -1.435869574546814,
 1.2248471975326538,
 -0.9610776305198669,
 -0.410573810338974,
 -0.25851747393608093,
 1.8455382585525513,
 0.551598072052002,
 0.1874181032180786,
 -0.5236652493476868,
 0.3692421317100525,
 0.5627151727676392,
 0.9747137427330017,
 -1.2993674278259277,
 -1.3465442657470703,
 0.638744592666626,
 0.0658125057

In [12]:
docs_score = db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='e86e26d8-b8c2-4aa2-a0a8-9119745c9cac', metadata={'source': 'speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between usâ€”however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='4ff56fbc-d94d-4009-ab77-667c7eb0a5c3', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that friendshipâ€”exercising

In [13]:
### Saving And Loading
db.save_local("faiss_index")

In [14]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)